# Chương 5 — Đồ thị tri thức và RAG: truy xuất lai BM25 + Dense + Graph trên y văn đái tháo đường

In [1]:
# RAG-00 · Thiết lập đường dẫn: mã chương dùng chung gói của tiểu luận (src/dm) và mã kế thừa của nhóm (src/tl01, tl03)
import os, sys, json, time, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
from pathlib import Path
warnings.filterwarnings("ignore")
TLF = Path.cwd().parent
for p in ["src", "src/tl01", "src/tl03"]:
    sys.path.insert(0, str(TLF / p))
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
print("Thư mục tiểu luận:", TLF)

Thư mục tiểu luận: /home/ngtukien/Projects/MultiAgent/tieuluan_final


Notebook khảo sát cấu trúc Đồ thị tri thức (KG) tiểu đường, thử nghiệm truy xuất BM25 và liên kết thực thể đồ thị, và tổng hợp kết quả đánh giá truy xuất / hỏi đáp có căn cứ.

In [2]:
# RAG-01 · Thống kê kho tài liệu và Đồ thị tri thức tiểu đường (96.188 tài liệu, 96.175 nút, 684.414 cạnh)
kb = json.loads((TLF / "src/tl03/outputs/metrics/ch5_kb_diabetes.json").read_text())
print(f"Số tài liệu: {kb['n_docs']:,d} (hồ sơ: {kb['n_records']:,d}, bản tổng hợp nhóm: {kb['n_summaries']})")
print(f"Số bộ ba (triples): {kb['n_triples']:,d} | Nút đồ thị: {kb['graph']['nodes']:,d} | Cạnh: {kb['graph']['edges']:,d}")
pd.Series(kb["graph"]["relations"], name="số cạnh").to_frame()

Số tài liệu: 96,188 (hồ sơ: 96,146, bản tổng hợp nhóm: 42)
Số bộ ba (triples): 1,357,436 | Nút đồ thị: 96,175 | Cạnh: 684,414


,số cạnh
has_gender,96146
has_smoking_history,96146
in_age_group,96146
bmi_class,96146
hba1c_class,96146
glucose_class,96146
diabetes_status,96146
has_condition,11384
risk_factor_of,5
indicates,2


In [3]:
# RAG-02 · Truy xuất BM25 trên kho MedQuAD-NIDDK (1.192 cặp hỏi–đáp của NIH) cho câu hỏi lâm sàng
from dm.ch7_medqa import KnowledgeTool
tool = KnowledgeTool()
hits = tool.search("What is the A1C threshold for diagnosing diabetes and prediabetes?", k=3)
for i, (title, snippet, score) in enumerate(hits, 1):
    print(f"[{i}] (BM25 = {score:.2f}) {title}\n    {snippet[:180]}...")

[1] (BM25 = 32.67) Diagnosis of Diabetes and Prediabetes — How to diagnose Diagnosis of Diabetes and Prediabetes ?
    Blood tests are used to diagnosis diabetes and prediabetes because early in the disease type 2 diabetes may have no symptoms. All diabetes blood tests involve drawing blood at a he...
[2] (BM25 = 29.89) Diagnosis of Diabetes and Prediabetes — What to do for Diagnosis of Diabetes and Prediabetes ?
    - Tests used for diagnosing diabetes and prediabetes include the A1C testfor type 2 diabetes and prediabetesthe fasting plasma glucose (FPG) test, and the oral glucose tolerance te...
[3] (BM25 = 29.07) Insulin Resistance and Prediabetes — How to diagnose Insulin Resistance and Prediabetes ?
    Health care providers use blood tests to determine whether a person has prediabetes, but they do not usually test specifically for insulin resistance. Insulin resistance can be ass...


In [4]:
# RAG-03 · Công cụ truy vấn chính xác trên Đồ thị tri thức tiểu đường (Domain.facts & Domain.link)
import pickle
with open(TLF / "src/tl03/models/ch5/diabetes_domain.pkl", "rb") as f:
    D = pickle.load(f)
q = "Tỷ lệ mắc tiểu đường của bệnh nhân nữ nhóm tuổi 60–79 có tăng huyết áp là bao nhiêu?"
print("Liên kết thực thể -> mã tài liệu:", D.link(q))
ans = D.facts(q)
print("Trả lời từ KG:", ans["answer"])
print("Nguồn / công cụ:", {k: v for k, v in ans.items() if k != "answer"})

Liên kết thực thể -> mã tài liệu: ['S-Female|60–79|1', 'S-gender=Female']
Trả lời từ KG: Nhóm nữ, nhóm tuổi 60–79, có tăng huyết áp có 1.830 hồ sơ; tỷ lệ mắc tiểu đường 31,42%, HbA1c trung bình 5,85%, đường huyết trung bình 152,1 mg/dL.
Nguồn / công cụ: {'tool': 'kg_aggregate', 'values': {'n': 1830, 'rate': 31.420765027322407}, 'evidence': ['giao các tập láng giềng vào của: age_group, gender, hypertension', '|tập hồ sơ| = 1830']}


In [5]:
# RAG-04 · So sánh hiệu năng các chiến lược truy xuất và xếp hạng lại trên KB tiếng Việt & MedQuAD
ret = json.loads((TLF / "src/tl02/outputs/metrics/diabetes.json").read_text()) if (TLF / "src/tl02/outputs/metrics/diabetes.json").exists() else {}
print("Thống kê KG lâm sàng (TL02):", json.loads((TLF / "src/tl02/outputs/metrics/kg.json").read_text()).get("summary", {}))

Thống kê KG lâm sàng (TL02): {}
